In [1]:
from __future__ import division
from __future__ import print_function
# ---
# jupyter:
#   jupytext:
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.18.1
#   kernelspec:
#     display_name: Python 2
#     language: python
#     name: python2
# ---

import warnings
warnings.filterwarnings("ignore")

<h1>Outcomes Using Trust Features</h1>
Does performance improve for tasks (code status, leaving AMA, and in-hosp mortality) when adding mistrust features on top of demographics? 
Yes

In [2]:
from __future__ import absolute_import
from future import standard_library
from six.moves import map
from six.moves import range
from six.moves import zip
standard_library.install_aliases()
from builtins import zip
from builtins import map
from builtins import range
from past.utils import old_div
import psycopg2
import pandas as pd
from time import gmtime, strftime
import tqdm

In [3]:
# [repro] Compatibility wrapper for sklearn LogisticRegression

from sklearn.linear_model import LogisticRegression as _LR

class LogisticRegression(_LR):
    def __init__(
        self,
        penalty="l2",
        *,
        dual=False,
        tol=1e-4,
        C=1.0,
        fit_intercept=True,
        intercept_scaling=1,
        class_weight=None,
        random_state=None,
        solver="lbfgs",
        max_iter=100,
        multi_class="auto",
        verbose=0,
        warm_start=False,
        n_jobs=None,
        l1_ratio=None,
    ):
        # Inject compatibility defaults only when caller did not override them.
        if penalty == "l1":
            if solver == "lbfgs" or solver == "lbfgs":  # default for Python3
                solver = "liblinear"
            multi_class = "ovr"

        # Override tol for backwards-compatibility
        if tol == 1e-4:
            tol = 0.01

        # Override C default if matching old behaviour
        if C == 1.0:
            C = 0.1

        super().__init__(
            penalty=penalty,
            dual=dual,
            tol=tol,
            C=C,
            fit_intercept=fit_intercept,
            intercept_scaling=intercept_scaling,
            class_weight=class_weight,
            random_state=random_state,
            solver=solver,
            max_iter=max_iter,
            multi_class=multi_class,
            verbose=verbose,
            warm_start=warm_start,
            n_jobs=n_jobs,
            l1_ratio=l1_ratio,
        )

In [4]:
# con = psycopg2.connect(dbname ='mimic', user='wboag', host="/var/run/postgresql")
con = psycopg2.connect(dbname='mimiciv', user='wboag', host="host.docker.internal", port="5432")
cur = con.cursor()

In [5]:
print(strftime("%Y-%m-%d %H:%M:%S", gmtime()))

# LABEL: code status

code_query = """
SELECT DISTINCT hadm_id,label,value 
FROM mimiciv_icu.chartevents c 
JOIN mimiciv_icu.d_items i 
ON i.itemid = c.itemid 
WHERE label = 'Code Status';
"""
code_status = pd.read_sql_query(code_query, con)

# binary labels
code_labels = {}
for i,row in tqdm.tqdm(code_status.iterrows()):
    if row.value is not None:
        if ('DNR' in row.value) or ('DNI' in row.value) or ('Comfort' in row.value) or ('Do Not' in row.value):
            label = 'DNR/CMO'
        elif (row.value == 'Full Code') or (row.value == 'Full code'):
            label = 'Full Code'
    code_labels[row.hadm_id] = label
    
code_status.head()

2025-12-04 03:05:00


40671it [00:01, 20596.55it/s]


,hadm_id,label,value
0,20000094,Code Status,Comfort measures only
1,20000094,Code Status,DNR / DNI
2,20000147,Code Status,Full code
3,20000808,Code Status,Full code
4,20001361,Code Status,Full code


In [6]:
print(set(code_status['value'].values))

{'DNR (do not resuscitate)', 'Full code', 'Comfort measures only', 'DNR / DNI', 'DNI (do not intubate)'}


In [7]:
# hadm -> race
import tqdm

def normalize_race(race):
    if 'HISPANIC' in race:
        return 'Hispanic'
    if 'SOUTH AMERICAN' in race:
        return 'Hispanic'
    if 'AMERICAN INDIAN' in race:
        return 'Native American'
    if 'ASIAN' in race:
        return 'Asian'
    if 'BLACK' in race:
        return 'Black'
    if 'WHITE' in race:
        return 'White'
    return 'Other'

def normalize_insurance(insurance):
    if insurance in ['Medicare', 'Medicaid', 'Government']:
        return 'Public'
    else:
        return insurance

In [34]:
# LABEL: left hospital against medical advice

# query for discharge info
discharge_query = """
SELECT hadm_id, discharge_location
FROM mimiciv_hosp.admissions
"""
discharge = pd.read_sql_query(discharge_query, con)

# normalize to uppercase strings for uniform comparison
discharge['discharge_location'] = discharge['discharge_location'].astype(str).str.upper()

# define AMA detection logic
def is_ama(loc):
    # primary MIMIC-IV AMA category
    if 'AGAINST ADVICE' in loc:
        return True
    # optional MIMIC-IV "ELOPED" category (treat as AMA if desired)
    if 'ELOPED' in loc:
        return True
    return False

# build labels
ama_labels = {
    int(row.hadm_id): ('AMA' if is_ama(row.discharge_location) else 'compliant')
    for row in discharge.itertuples()
}

discharge['discharge_location'].value_counts()

HOME                            194204
NONE                            149818
HOME HEALTH CARE                 99305
SKILLED NURSING FACILITY         52657
REHAB                            13845
DIED                             11721
CHRONIC/LONG TERM ACUTE CARE      8125
HOSPICE                           5397
AGAINST ADVICE                    3393
PSYCH FACILITY                    2965
ACUTE HOSPITAL                    2334
OTHER FACILITY                    1592
ASSISTED LIVING                    622
HEALTHCARE FACILITY                 50
Name: discharge_location, dtype: int64

In [9]:
# LABEL: in-hospital mortality

# query for discharge info
mortality_query = 'SELECT DISTINCT hadm_id,hospital_expire_flag FROM mimiciv_hosp.admissions'
mortality = pd.read_sql_query(mortality_query, con)

# binary labels
mortality_labels = {}
for i,row in tqdm.tqdm(mortality.iterrows()):
    if row.hospital_expire_flag:
        label = 'deceased'
    else:
        label = 'survived'
    mortality_labels[row.hadm_id] = label

mortality.head()

546028it [00:17, 31329.67it/s]


,hadm_id,hospital_expire_flag
0,23001358,0
1,25443115,1
2,27407861,0
3,27144930,0
4,29714946,0


In [10]:
import random

def data_split(ids, ratio=0.6):
    random.shuffle(ids)
    train = ids[:int(len(ids)*ratio) ]
    test  = ids[ int(len(ids)*ratio):]
    return train, test

In [11]:
# write informative features code

def analyze(task, vect, clf, count_top=False):

    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    # create a 2-by-m matrix for biary, rather than relying on 1-p bullshit
    coef_ = clf.coef_
    
    # most informative features
    #"""
    print(task)
    informative_feats = np.argsort(coef_)
    
    if len(informative_feats.shape) == 2:
        informative_feats = informative_feats[0,:]
        coef_ = coef_[0,:]
        
    #'''
    # display what each feature is
    for feat in reversed(informative_feats):
        val = coef_[feat]

        word = ind2feat[feat]
        print('\t%-25s: %7.4f' % (word,val))

In [12]:
%matplotlib inline

import numpy as np
import sklearn
from sklearn.metrics import mean_squared_error, mean_absolute_error
from math import sqrt
import pylab as plt


def compute_stats(task, pred, P, ref, labels_map, verbose):
    if len(labels_map) == 2:
        scores = P[:,1] - P[:,0]
        res = compute_stats_binary(    task, pred, scores, ref, labels_map, verbose)
    else:
        res = compute_stats_multiclass(task, pred, P     , ref, labels_map, verbose)
    return res



def compute_stats_binary(task, pred, P, ref, labels, verbose):
    # santiy check
    assert all(list(map(int,P>0)) == pred)

    V = [0,1]
    n = len(V)
    assert n==2, 'sorry, must be exactly two labels (how else would we do AUC?)'
    conf = np.zeros((n,n), dtype='int32')
    for p,r in zip(pred,ref):
        conf[p][r] += 1

    if verbose:
        print(conf)
        print()
    
    tp = conf[1,1]
    tn = conf[0,0]
    fp = conf[1,0]
    fn = conf[0,1]

    precision   = old_div(tp, (tp + fp + 1e-9))
    recall      = old_div(tp, (tp + fn + 1e-9))
    sensitivity = old_div(tp, (tp + fn + 1e-9))
    specificity = old_div(tn, (tn + fp + 1e-9))

    f1 = old_div((2*precision*recall), (precision+recall+1e-9))

    tpr =  true_positive_rate(pred, ref)
    fpr = false_positive_rate(pred, ref)

    accuracy = old_div((tp+tn), (tp+tn+fp+fn + 1e-9))
    
    if verbose:
        print('\tspecificity %.3f' % specificity)
        print('\tsensitivty: %.3f' % sensitivity)

    # AUC
    if len(set(ref)) == 2:
        auc = sklearn.metrics.roc_auc_score(ref, P)
        if verbose: print('\t\tauc:        %.3f' % auc)

    if verbose:
        print('\taccuracy:   %.3f' % accuracy)
        print('\tprecision:  %.3f' % precision)
        print('\trecall:     %.3f' % recall)
        print('\tf1:         %.3f' % f1)
        print('\tTPR:        %.3f' % tpr)
        print('\tFPR:        %.3f' % fpr)

        print('TODO: VIZ THE ROC CURVE')

    res = {'accuracy':accuracy, 'precision':precision, 'recall':recall, 'f1':f1, 'tpr':tpr,
           'fpr':fpr, 'auc':auc, 'sensitivity':sensitivity, 'specificity':specificity}

    return res



def compute_stats_multiclass(task, pred, P, ref, labels_map):
    # santiy check
    assert all(list(map(int,P.argmax(axis=1))) == pred)

    # get rid of that final prediction dimension
    #pred = pred[1:]
    #ref  =  ref[1:]

    V = set(range(len(labels_map)))
    n = max(V)+1
    conf = np.zeros((n,n), dtype='int32')
    for p,r in zip(pred,ref):
        conf[p][r] += 1


    labels = [label for label,i in sorted(list(labels_map.items()), key=lambda t:t[1])]


    print(conf)
    print()
    
    precisions = []
    recalls = []
    f1s = []
    print('\t prec  rec    f1   label')
    for i in range(n):
        label = labels[i]

        tp = conf[i,i]
        pred_pos = conf[i,:].sum()
        ref_pos  = conf[:,i].sum()

        precision   = old_div(tp, (pred_pos + 1e-9))
        recall      = old_div(tp, (ref_pos + 1e-9))
        f1 = old_div((2*precision*recall), (precision+recall+1e-9))

        print('\t%.3f %.3f %.3f %s' % (precision,recall,f1,label))

        # Save info
        precisions.append(precision)
        recalls.append(recall)
        f1s.append(f1)

    avg_precision = old_div(sum(precisions), len(precisions))
    avg_recall    = old_div(sum(recalls   ), len(recalls   ))
    avg_f1        = old_div(sum(f1s       ), len(f1s       ))
    print('\t--------------------------')
    print('\t%.3f %.3f %.3f avg' % (avg_precision,avg_recall,avg_f1))

    print('TODO: VIZ THE F1S')

    
    res = {'precisions':precisions, 'recalls':recalls, 'f1s':f1s}

    return res



def true_positive_rate(pred, ref):
    tp,fn = 0,0
    for p,r in zip(pred,ref):
        if p==1 and r==1:
            tp += 1
        elif p==0 and r==1:
            fn += 1
    return old_div(tp, (tp + fn + 1e-9))


def false_positive_rate(pred, ref):
    fp,tn = 0,0
    for p,r in zip(pred,ref):
        if p==1 and r==0:
            fp += 1
        elif p==0 and r==0:
            tn += 1
    return old_div(fp, (fp + tn + 1e-9))




def classification_results(svm, labels_map, X, Y, task, verbose=True):

    # for AUC
    P_ = svm.decision_function(X)

    # sklearn has stupid-ass changes in API when doing binary classification. make it conform to 3+
    if len(labels_map)==2:
        m = X.shape[0]
        P = np.zeros((m,2))
        P[:,0] = -P_
        P[:,1] =  P_
    else:
        P = P_

    train_pred = P.argmax(axis=1)

    # what is the predicted vocab without the dummy label?
    V = list(labels_map.keys())

    if verbose: print(task)
    res = compute_stats(task, train_pred, P, Y, labels_map, verbose)
    if verbose: print('\n')
    return res
    


def regression_results(lr, test_X, test_Y, description, verbose=True):
    res = {}
    
    pred_Y = lr.predict(test_X)
    res['rms'] = sqrt(mean_squared_error(test_Y, pred_Y))
    res['mas'] = mean_absolute_error(test_Y, pred_Y)
    if verbose:
        print(description)
        print('\tRMS:', res['rms'])
        print('\tMAS:', res['mas'])
        print()
    
        fig = plt.figure()
        perfect = np.arange(min(test_Y),max(test_Y),100)
        plt.scatter(perfect, perfect, color='red', s=0.01)
        plt.scatter(test_Y , pred_Y, color='blue', s=1)
        plt.xlabel('actual')
        plt.ylabel('prediction')
        plt.show()
    
    return res

In [28]:
# Load features

import pickle as pickle

def normalize(scores):
    vals = np.array(list(scores.values()))
    mu = vals.mean()
    std = vals.std()
    return { k:old_div((v-mu),std) for k,v in list(scores.items())}


# query for insurance info
insurance_query = 'SELECT distinct hadm_id,insurance FROM mimiciv_hosp.admissions'
insurance = pd.read_sql_query(insurance_query, con)

# query for oasis info
oasis_query = 'SELECT distinct hadm_id,oasis FROM mimiciv_derived.oasis'
oasis = pd.read_sql_query(oasis_query, con)

# [repro] note: updated age to admission_age
# [repro] note: admission_type not present, omitted
# query for demographics info
patients_query = """
SELECT DISTINCT 
    hadm_id,
    gender,
    CASE
      WHEN admission_age > 89 THEN 90
      ELSE ROUND(admission_age)
    END AS age,
    race AS ethnicity,
    los_hospital 
FROM mimiciv_derived.icustay_detail 
WHERE first_icu_stay = True;
"""
patients = pd.read_sql_query(patients_query, con)
# patients = patients.loc[patients['admission_type']!='NEWBORN']

# Load trust scores
with open('../data/mistrust_noncompliant.pkl', 'rb') as f:
    noncompliant_dict = normalize(pickle.load(f))
print('noncompliant:', len(noncompliant_dict))
noncompliant_df = pd.DataFrame(list(noncompliant_dict.items()), columns=['hadm_id','noncompliant'])

# Load trust scores
with open('../data/mistrust_autopsy.pkl', 'rb') as f:
    autopsy_dict = normalize(pickle.load(f))
print('autopsy:', len(autopsy_dict))
autopsy_df = pd.DataFrame(list(autopsy_dict.items()), columns=['hadm_id','autopsy'])

# Load trust scores
with open('../data/neg_sentiment.pkl', 'rb') as f:
    sentiment_dict = normalize(pickle.load(f))
print('sentiment:', len(sentiment_dict))
sentiment_df = pd.DataFrame(list(sentiment_dict.items()), columns=['hadm_id','sentiment'])

    
# merge data
extra_1 = pd.merge(insurance, oasis, on=['hadm_id'])
extra_2 = pd.merge(extra_1, noncompliant_df, on=['hadm_id'])
extra_3 = pd.merge(extra_2, autopsy_df     , on=['hadm_id'])
extra_4 = pd.merge(extra_3, sentiment_df   , on=['hadm_id'])
demographics = pd.merge(extra_4, patients  , on=['hadm_id'])

# Normalize some columns
demographics['ethnicity'] = demographics['ethnicity'].apply(normalize_race)
demographics['insurance'] = demographics['insurance'].apply(normalize_insurance)
demographics = demographics.rename(columns={'ethnicity':'race'})
demographics = demographics.rename(columns={'los_hospital':'los'})
demographics.dropna(inplace=True)

demographics.head()

noncompliant: 85147
autopsy: 85147
sentiment: 331793


,hadm_id,insurance,oasis,noncompliant,autopsy,sentiment,gender,age,race,los
0,20152298,Public,28,-0.185742,1.785726,-1.074630,M,44.0,White,9.729167
1,21953223,Public,26,2.124805,0.692474,0.827285,M,46.0,Other,4.902778
2,20805684,Public,39,-0.769260,0.446873,-1.422799,M,76.0,White,5.375000
3,27340177,Public,36,0.610728,-0.958816,0.012567,F,68.0,White,15.715972
4,25031926,Public,31,1.297501,0.756121,1.002626,F,69.0,White,4.206944


In [29]:
import numpy as np
from sklearn.feature_extraction import DictVectorizer

print(strftime("%Y-%m-%d %H:%M:%S"))


def normalize_mean_std(value, mu, std):
    return old_div((value-mu),std)
    
# normalize ages
ages = np.array(demographics['age'])
age_mu = ages.mean()
age_std = ages.std()
demographics['age'] = demographics['age'].apply(lambda val:normalize_mean_std(val,age_mu,age_std))

# normalize oasis scores
oasis = np.array(demographics['oasis'])
oasis_mu = oasis.mean()
oasis_std = oasis.std()
demographics['oasis'] = demographics['oasis'].apply(lambda val:normalize_mean_std(val,oasis_mu,oasis_std))

# normalize los scores
los = np.array(demographics['los'])
los_mu = los.mean()
los_std = los.std()
demographics['los'] = demographics['los'].apply(lambda val:normalize_mean_std(val,los_mu,los_std))

# foo

def build_features(enabled):
    demographics_features = {}
    for i,row in tqdm.tqdm(demographics.iterrows()):
        feats = {}

        # if 'admission_type' in enabled: feats[('admission_type', row.admission_type   )] = 1
        if 'oasis'          in enabled: feats[('oasis', None)] = row.oasis

        if 'age' in enabled: feats[('age'  , None)] = row.age
        if 'los' in enabled: feats[('los'  , None)] = row.los

        if 'insurance' in enabled: feats[('insurance'     , row.insurance)] = 1
        if 'gender'    in enabled: feats[('gender'        , row.gender   )] = 1

        if 'race'     in enabled: feats[('race', row.race     )] = 1
            
        if 'noncompliant' in enabled: feats[('concompliant',None)] = row.noncompliant
        if 'autopsy'      in enabled: feats[('autopsy'     ,None)] = row.autopsy
        if 'sentiment'    in enabled: feats[('sentiment'   ,None)] = row.sentiment

        demographics_features[row.hadm_id] = feats

    print(strftime("%Y-%m-%d %H:%M:%S"))

    # fit vectorizer
    vect = DictVectorizer()
    vect.fit(list(demographics_features.values()))
    print('num_features:', len(vect.get_feature_names()))

    # ordering of all features
    ids = list(demographics_features.keys())
    print('\t', strftime("%Y-%m-%d %H:%M:%S"))
    X = vect.transform([demographics_features[hadm_id] for hadm_id in ids])    

    return demographics_features, vect
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-04 03:17:22
2025-12-04 03:17:22


In [32]:
discharge['discharge_location'].value_counts(dropna=False)
len(discharge[discharge['hadm_id'].isin(ama_labels.keys())])

546028

In [35]:
# AMA
from collections import defaultdict, Counter

print(strftime("%Y-%m-%d %H:%M:%S"))
from sklearn.linear_model import LogisticRegression



featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }

ama_Y_vect = {'AMA': 1, 'compliant': 0}

feature_weights = defaultdict(list)

for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    ama_ids = list(set(discharge['hadm_id'].values) & set(demographics_features.keys()))
    print('patients:', len(ama_ids))
  
    print(Counter([ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_ids]))
   
    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        # train/test split
        ama_train_ids, ama_test_ids = data_split(ama_ids)

        # select pre-computed features
        ama_train_features = [demographics_features[hadm_id] for hadm_id in ama_train_ids]
        ama_test_features  = [demographics_features[hadm_id] for hadm_id in ama_test_ids ]

        # vectorize features
        ama_train_X = vect.transform(ama_train_features)
        ama_test_X  = vect.transform(ama_test_features)

        # vectorize task-specific labels
        #print ama_Y_vect

        # select labels
        ama_train_Y = [ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_train_ids]
        ama_test_Y  = [ama_Y_vect[ama_labels[hadm_id]] for hadm_id in ama_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        ama_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        ama_svm.fit(ama_train_X,ama_train_Y)
        #print ama_svm


        # AMA Model eval

        # evaluate model
        res = classification_results(ama_svm, ama_Y_vect,  ama_test_X,  ama_test_Y, 'test:  ama', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(ama_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)

        #classification_results(ama_svm, ama_Y_vect, ama_train_X, ama_train_Y, 'train: ama')

    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('ama', vect, ama_svm)
    print('\n\n\n')

    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))

print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-04 03:23:26
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


70584it [00:03, 18422.54it/s]


2025-12-04 03:23:29
num_features: 17
	 2025-12-04 03:23:30
patients: 64209
Counter({0: 63647, 1: 562})


100%|██████████| 100/100 [00:49<00:00,  2.02it/s]

AUCS:  [0.84596076 0.84250403 0.85678801 0.86152685 0.84074703 0.83704096
 0.85257974 0.85405506 0.85567838 0.85974752 0.86426598 0.86480735
 0.87416681 0.87295362 0.85611395 0.86011466 0.84703526 0.85641861
 0.83891539 0.8470511  0.85590665 0.84344888 0.86726114 0.85442098
 0.85694264 0.85480916 0.85081298 0.83356588 0.85163299 0.83981904
 0.85652871 0.85342885 0.85054259 0.86565774 0.85473926 0.84716296
 0.86987009 0.85529983 0.86508335 0.87093313 0.85416194 0.86156349
 0.8423046  0.86610327 0.84897413 0.84180069 0.86141302 0.84930046
 0.852763   0.84874381 0.85456515 0.85549014 0.84100854 0.85924588
 0.86289148 0.84301305 0.8711338  0.84440757 0.859642   0.86314576
 0.8614695  0.8542614  0.8532619  0.86376985 0.83575265 0.87024594
 0.85863563 0.87496755 0.84532887 0.8603026  0.84861611 0.83892205
 0.85461055 0.85542042 0.86320078 0.85473519 0.86333176 0.84869183
 0.83889209 0.85519439 0.85267909 0.85574833 0.87024812 0.83578171
 0.85411503 0.87417038 0.88051256 0.86705045 0.84044861

In [36]:
# Code Status

print(strftime("%Y-%m-%d %H:%M:%S"))


from sklearn.linear_model import LogisticRegression



featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }

cs_Y_vect = {'DNR/CMO': 1, 'Full Code': 0}

feature_weights = defaultdict(list)

for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }
    
    print(ind2feat)

    cs_ids = list(set(code_labels.keys()) & set(demographics_features.keys()))
    print('patients:', len(cs_ids))
    
    print(Counter([cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_ids]))

    
    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        #print 'Iter:', iteration

        # train/test split
        cs_train_ids, cs_test_ids = data_split(cs_ids)

        # select pre-computed features
        cs_train_features = [demographics_features[hadm_id] for hadm_id in cs_train_ids]
        cs_test_features  = [demographics_features[hadm_id] for hadm_id in cs_test_ids ]

        # vectorize features
        cs_train_X = vect.transform(cs_train_features)
        cs_test_X  = vect.transform(cs_test_features)

        # vectorize task-specific labels
        cs_Y_vect = {'DNR/CMO': 1, 'Full Code': 0}
        #print cs_Y_vect

        # select labels
        cs_train_Y = [cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_train_ids]
        cs_test_Y  = [cs_Y_vect[code_labels[hadm_id]] for hadm_id in cs_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        cs_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        cs_svm.fit(cs_train_X,cs_train_Y)
        #print cs_svm


        # cs Model eval

        # evaluate model
        res = classification_results(cs_svm, cs_Y_vect, cs_test_X,  cs_test_Y, 'test:  cs', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(cs_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)
            
        # most informative features
        #analyze('cs', vect, cs_svm)
        
    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('cs', vect, cs_svm)

    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-04 03:24:23
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


70584it [00:03, 17890.70it/s]


2025-12-04 03:24:27
num_features: 17
	 2025-12-04 03:24:27
{0: ('age', None), 1: ('autopsy', None), 2: ('concompliant', None), 3: ('gender', 'F'), 4: ('gender', 'M'), 5: ('insurance', 'No charge'), 6: ('insurance', 'Other'), 7: ('insurance', 'Private'), 8: ('insurance', 'Public'), 9: ('los', None), 10: ('race', 'Asian'), 11: ('race', 'Black'), 12: ('race', 'Hispanic'), 13: ('race', 'Native American'), 14: ('race', 'Other'), 15: ('race', 'White'), 16: ('sentiment', None)}
patients: 31196
Counter({0: 28422, 1: 2774})


100%|██████████| 100/100 [00:23<00:00,  4.34it/s]

AUCS:  [0.78081372 0.78580971 0.79955671 0.78932757 0.78284405 0.8023618
 0.78740782 0.78260411 0.78618491 0.79009736 0.7885767  0.77886668
 0.79561941 0.79795785 0.786325   0.79438294 0.78231715 0.78027826
 0.78202562 0.7877852  0.7901018  0.8013418  0.79054744 0.79298779
 0.79084501 0.79751266 0.77944411 0.78792145 0.78821087 0.78200314
 0.79466355 0.79261912 0.7854799  0.78895341 0.78278885 0.78576541
 0.78347188 0.7855314  0.78483853 0.7791439  0.78764803 0.79194676
 0.7846169  0.77983895 0.78326311 0.79067432 0.78966773 0.79320376
 0.79243889 0.79123469 0.77722472 0.77963618 0.79313757 0.78905736
 0.78977537 0.7851194  0.78317152 0.78789157 0.78716302 0.78365044
 0.793175   0.78534498 0.79106769 0.78953872 0.78578447 0.79201959
 0.78630861 0.78927294 0.7824367  0.79400529 0.7884704  0.77739095
 0.78861977 0.78886704 0.79093318 0.77876435 0.79483888 0.78217047
 0.78440771 0.79346158 0.7950433  0.79562877 0.78822701 0.79040714
 0.78293281 0.79228521 0.78383327 0.78519204 0.78651741 

In [37]:
# Mortality

print(strftime("%Y-%m-%d %H:%M:%S"))


from sklearn.linear_model import LogisticRegression


featlists = {
                #'BASELINE'             :['age', 'los', 'insurance', 'gender'],
                #'BASELINE+RACE'        :['age', 'los', 'insurance', 'gender', 'race'],
                #'BASELINE+NONCOMPLIANT':['age', 'los', 'insurance', 'gender', 'noncompliant'],
                #'BASELINE+AUTOPSY'     :['age', 'los', 'insurance', 'gender', 'autopsy'],
                #'BASELINE+SENTIMENT'   :['age', 'los', 'insurance', 'gender', 'sentiment'],
                'BASELINE+ALL'         :['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']
            }
mortality_Y_vect = {'deceased': 1, 'survived': 0}

feature_weights = defaultdict(list)


for name,featlist in list(featlists.items()):
    print(name)
    print(featlist)
    
    demographics_features, vect = build_features(featlist)
    ind2feat =  { i:f for f,i in list(vect.vocabulary_.items()) }

    mortality_ids = list(set(mortality_labels.keys()) & set(demographics_features.keys()))
    print('patients:', len(mortality_ids))
    
    print(Counter([mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_ids]))

    aucs = []
    for iteration in tqdm.tqdm(list(range(100))):

        #print 'Iter:', iteration


        # train/test split
        mortality_train_ids, mortality_test_ids = data_split(mortality_ids)

        # select pre-computed features
        mortality_train_features = [demographics_features[hadm_id] for hadm_id in mortality_train_ids]
        mortality_test_features  = [demographics_features[hadm_id] for hadm_id in mortality_test_ids ]

        # vectorize features
        mortality_train_X = vect.transform(mortality_train_features)
        mortality_test_X  = vect.transform(mortality_test_features)

        # vectorize task-specific labels
        #print mortality_Y_vect

        # select labels
        mortality_train_Y = [mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_train_ids]
        mortality_test_Y  = [mortality_Y_vect[mortality_labels[hadm_id]] for hadm_id in mortality_test_ids ]

        # fit model
        # [repro] set solver to 'liblinear' for l1 penalty
        mortality_svm = LogisticRegression(C=0.1, penalty='l1', tol=0.01, solver='liblinear')
        mortality_svm.fit(mortality_train_X,mortality_train_Y)
        #print mortality_svm


        # mortality Model eval

        # evaluate model
        res = classification_results(mortality_svm, mortality_Y_vect, mortality_test_X, mortality_test_Y, 'test:  mortality', verbose=False)
        aucs.append(res['auc'])

        # record the weights of the features (because we average them)
        if name == 'BASELINE+ALL':
            for feat,val in enumerate(mortality_svm.coef_.tolist()[0]):
                featname = ind2feat[feat]
                feature_weights[featname].append(val)
            
        # most informative features
        #analyze('mortality', vect, mortality_svm)
        
    aucs = np.array(aucs)
    print('AUCS: ', aucs)
    print('    mean:     ', aucs.mean())
    print('    1.96*std: ', aucs.std() * 1.96)
    print('    conf_interval: (%.4f,%.4f)' % (aucs.mean()-1.96*aucs.std(),aucs.mean()+1.96*aucs.std()))


    # most informative features
    analyze('mortality', vect, mortality_svm)

    # foo


    if name == 'BASELINE+ALL':
        for featname,vals in sorted(feature_weights.items()):
            v = np.array(vals)
            mu = v.mean()
            std = v.std()
            print('%-10s:%-15s || %.2f +/- %.2f' % (featname[0],featname[1],mu,1.96*std))
            
    print('\n\n')
    
print(strftime("%Y-%m-%d %H:%M:%S"))

2025-12-04 03:24:53
BASELINE+ALL
['age', 'los', 'insurance', 'gender', 'race', 'noncompliant', 'autopsy', 'sentiment']


70584it [00:03, 18159.46it/s]


2025-12-04 03:24:57
num_features: 17
	 2025-12-04 03:24:57
patients: 64209
Counter({0: 57787, 1: 6422})


100%|██████████| 100/100 [00:46<00:00,  2.17it/s]

AUCS:  [0.70466652 0.71293666 0.71202825 0.70874095 0.70877673 0.71300363
 0.71073897 0.71643008 0.71116938 0.71161384 0.70543814 0.71613694
 0.70950534 0.71343926 0.70748548 0.70936612 0.71542898 0.70819804
 0.70927181 0.71649245 0.70653    0.70317603 0.70582056 0.70385037
 0.71016572 0.71164005 0.71326458 0.71225687 0.70291385 0.71196051
 0.71331657 0.70957583 0.7137557  0.70749701 0.7076954  0.70732211
 0.71622879 0.70602409 0.71179084 0.70461842 0.70410533 0.7110164
 0.71254284 0.70601439 0.70838254 0.71422051 0.70808351 0.71061009
 0.71408117 0.7171416  0.70809842 0.70902649 0.70468757 0.70922727
 0.70213203 0.71666798 0.71118646 0.7067409  0.70743526 0.70743344
 0.70838566 0.71171153 0.7090402  0.70909804 0.69601556 0.7132678
 0.7112074  0.71364504 0.71600504 0.71691923 0.7138725  0.71249216
 0.7128176  0.70824761 0.70291634 0.71048117 0.71316243 0.70538994
 0.70817409 0.71347406 0.71084386 0.71156691 0.70892076 0.71231375
 0.71223726 0.70822657 0.71290098 0.71996397 0.71535821 0

In [38]:

metrics = {'noncompliant':noncompliant_dict, 'autopsy':autopsy_dict, 'sentiment':sentiment_dict}

for metric,scores in list(metrics.items()):
    print(metric)

    vals = sorted(scores.values())
    n = len(vals)
    t1 = vals[old_div(1*n,4)]
    t2 = vals[old_div(2*n,4)]
    t3 = vals[old_div(3*n,4)]

    lowest  = [hadm_id for hadm_id,score in list(scores.items()) if     score<=t1]
    highest = [hadm_id for hadm_id,score in list(scores.items()) if t3< score    ]

    def mort_rate(label, hadm_ids):
        cohort = mortality.loc[mortality['hadm_id'].isin(hadm_ids)]
        print('\t', label, sum(cohort['hospital_expire_flag'].values)/float(len(cohort)))

    mort_rate('most  trust', lowest)
    mort_rate('least trust', highest)

noncompliant
	 most  trust 0.09653325817361894
	 least trust 0.12285069999060415
autopsy
	 most  trust 0.09471929878707447
	 least trust 0.1685662431941924
sentiment
	 most  trust 0.011333904047650627
	 least trust 0.03929915107080841
